In [ ]:
import os
import re
import json
import pandas as pd
import requests
from datetime import datetime


CSV_PATH = "/content/cloze_items_combined_100percondition_with_DI_MIS.csv"
MODEL_NAME = "YOUR MODEL NAME
SENT_COL = "Sentence"
COND_COL = "Condition"

OLLAMA_CHAT_URL = "http://127.0.0.1:11434/api/chat"
OLLAMA_GEN_URL  = "http://127.0.0.1:11434/api/generate"

TEMPERATURE = 0.1
TOP_P = 1
TUR_SAYISI = 3


timestamp = datetime.now().strftime("%Y%m%d_%H%M")
safe_model = MODEL_NAME.replace(":", "_").replace("/", "_")
BASE_DIR = "/content"
OUT_PATH = os.path.join(BASE_DIR, f"{safe_model}-V1-{timestamp}.csv")
DEBUG_JSONL = os.path.join(BASE_DIR, f"{safe_model}-V1-{timestamp}.debug.jsonl")


BLANK_RE = re.compile(r"_{3,}")

def extract_prefix(sentence: str):
    s = str(sentence)
    m = BLANK_RE.search(s)
    if not m:
        return None
    prefix = s[:m.start()].rstrip()
    return prefix

def clean_fill(fill: str):
    t = (fill or "").strip()
    t = t.split("\n")[0].strip()
    t = t.strip().strip('"').strip("'")

    if len(t.split()) > 15:
        t = " ".join(t.split()[:15])
    return t


def build_prompt(prefix: str) -> str:

    return (
        'Aşağıdaki cümlede "__________." ile belirtilmiş bir boşluk var. '
        "Bu boşluğu cümlenin yapısının akışına en uygun fiille tamamla.\n\n"
        "Cevap verirken kurallar:\n"
        "- Sadece boşluğa gelecek kısmı yaz.\n"
        "- Cümleyi tekrar etme.\n"
        "- Açıklama yapma.\n"
        "Tek kelime cevap ver ve fiil kullanarak tamamla"
        '- Cevap vermeden geçme.\n\n'
        f"Cümle:\n{prefix} __________.\n"
        "Cevap:"
    )


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

@torch.no_grad()
def _hf_generate(prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    out_ids = model.generate(
        **inputs,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0,
        eos_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else None,
    )

    full = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    gen = full[len(prompt):].strip() if full.startswith(prompt) else full.strip()
    return gen, {"backend": "transformers", "model": MODEL_NAME}


def ollama_chat(prompt: str):
    content, meta = _hf_generate(prompt)

    return content, {"message": {"content": content}, "meta": meta}

def ollama_generate(prompt: str):
    content, meta = _hf_generate(prompt)

    return content, {"response": content, "meta": meta}

def save_debug(line_obj: dict):
    with open(DEBUG_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(line_obj, ensure_ascii=False) + "\n")


df = pd.read_csv(os.path.join(BASE_DIR, CSV_PATH))
df = df[df[COND_COL].isin(["High", "Low"])].reset_index(drop=True)


rows = []
for i, row in df.iterrows():
    sent = str(row[SENT_COL])
    rows.append({
        "Condition": row[COND_COL],
        "Sentence": sent,
        "Model": MODEL_NAME,
        "temperature": TEMPERATURE,
        "top_p": TOP_P
    })

out_df = pd.DataFrame(rows)

empty_count = 0


for t in range(1, TUR_SAYISI + 1):

    tur_outputs = []

    print(f"\n========== TUR {t} BAŞLADI ==========")

    for i, row in df.iterrows():
        sent = str(row[SENT_COL])
        prefix = extract_prefix(sent)

        if prefix is None:
            tur_outputs.append("")
            continue

        prompt = build_prompt(prefix)


        resp, meta = ollama_chat(prompt)
        fill = clean_fill(resp)


        if not fill:
            resp2, meta2 = ollama_generate(prompt)
            fill = clean_fill(resp2)

            save_debug({
                "i": i,
                "tur": t,
                "condition": row[COND_COL],
                "prefix": prefix,
                "prompt": prompt,
                "chat_json": meta,
                "generate_json": meta2
            })

        if not fill:
            empty_count += 1
            print(f"[TUR {t}][{i+1}/{len(df)}] BOŞ! ({row[COND_COL]}) | {prefix[-60:]}")
        else:
            print(f"[TUR {t}][{i+1}/{len(df)}] {row[COND_COL]} -> {fill!r}")

        tur_outputs.append(fill)

    out_df[f"tur{t}"] = tur_outputs

out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print("\nKAYDEDİLDİ ->", OUT_PATH)
print("DEBUG JSONL ->", DEBUG_JSONL)
print("Boş completion sayısı ->", empty_count)
